In [0]:
"""
Number of Comments Per User in 30 days before 2020-02-10


Last Updated: April 2026

Easy
ID 2004

197

Return the total number of comments received for each user in the 30-day period up to and including 2020-02-10. Don't output users who haven't received any comment in the defined time period."""

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, sum as spark_sum
from pyspark.sql.types import StructType, StructField, IntegerType, DateType

spark = SparkSession.builder.appName("CommentsLast30Days").getOrCreate()

# Dummy data: user_id, created_at, number_of_comments
data = [
    (101, "2020-01-12", 5),
    (102, "2020-01-15", 3),
    (101, "2020-01-20", 2),
    (103, "2020-01-25", 8),
    (101, "2020-02-01", 4),
    (102, "2020-02-05", 1),
    (104, "2020-02-08", 6),
    (103, "2020-02-09", 3),
    (101, "2020-02-10", 2),      # on the target date
    (105, "2019-12-01", 10),     # outside 30-day window (too old)
    (106, "2020-02-11", 7),      # after target date (excluded)
]

# Create DataFrame
df = spark.createDataFrame(data, ["user_id", "created_at", "number_of_comments"])
df = df.withColumn("created_at", to_date(col("created_at")))

df.createOrReplaceTempView("fb_comments_count")

# Preview
df.show()

In [0]:
spark.sql("""
    select user_id, sum(number_of_comments) as total_comments
    from fb_comments_count
    where created_at between date_sub('2020-02-10', 30) AND '2020-02-10'
    group by user_id
""").display()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, date_sub

# Define target date
target_date = "2020-02-10"

# Filter for last 30 days including target date
filtered_df = df.filter(
    (col("created_at") >= date_sub(target_date, 30)) & 
    (col("created_at") <= target_date)
)

# Aggregate by user
result = filtered_df.groupBy("user_id").agg(
    spark_sum("number_of_comments").alias("total_comments")
).filter(
    col("total_comments") > 0
).orderBy("user_id")

# Show results
result.show()